## XGBoost v. NN baseline

**Config**: n_train at 100k, n_test at 10k, $\sigma \in [0.1, 0.5],$ with $r, T$ default values. All hyperparameters at default.

This experiment determines the performance (RMSE, MAE) of both a neural network and a standard XGBoost model across log moneyness regions. The standard regions used are as follows:
- ITM: $lm = \log \left ( \frac{S}{K} \right ) \in [0.5, 2.0)$
- ATM: $lm \in [-0.2, 0.2]$
- OTM: $lm \in [-2, -0.5]$

In [1]:
import sys, os

sys.path.append("..")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from src.data.dataset_generator import DataConfig, generate_bundle

# data generation code
base_cfg = DataConfig(
    n_train=100_000,
    n_test=10_000
    )

regions = {
    "ATM": (-0.2, 0.2),
    "ITM": (0.5, 2.0),
    "OTM": (-2, -0.5)
}

bundles = {
}

for region, lm_range in regions.items():
    cfg = base_cfg.replace(log_moneyness_range=lm_range)
    bundles[region] = generate_bundle(cfg)

### Hyperparameter tuning

Small grid search on a standard-distribution bundle (`log_m ∈ [-2, 2]`). We pick the config with lowest validation MAE for each model, then reuse those winners across all regimes below.

- XGBoost grid: `max_depth ∈ {5, 7}`, `learning_rate ∈ {0.03, 0.1}`, `n_estimators ∈ {500, 1000}` — 8 configs
- NN grid: `lr ∈ {1e-3, 3e-4}`, `loss_alpha ∈ {0.2, 0.5, 0.8}` — 6 configs

We start with tuning the NN, then the XGB.

In [ ]:
import itertools
import numpy as np
import pandas as pd

from src.data.dataset_generator import generate_bundle
from src.training.train_xgboost import train as xgb_train, predict as xgb_predict
from src.training.train_nn import train as nn_train, predict as nn_predict, set_seed, get_device
from src.eval.validate import compute_metrics


def val_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true)))


tuning_cfg = base_cfg  # base_cfg has log_moneyness_range default = (-2, 2)
tuning_bundle = generate_bundle(tuning_cfg)
Xt, yt = tuning_bundle.train.X, tuning_bundle.train.y_call
Xv, yv = tuning_bundle.val.X,   tuning_bundle.val.y_call

device = get_device(use_cpu=True)
nn_grid = list(itertools.product([1e-3, 3e-4], [0.2, 0.5, 0.8]))
nn_trials = []
for lr, loss_alpha in nn_grid:
    print(f"Training NN: lr={lr}, loss_alpha={loss_alpha}", flush=True)

    set_seed(42)
    params = {"lr": lr, "loss_alpha": loss_alpha}
    model, np_ = nn_train(Xt, yt, Xv, yv, device=device, patience=10, epochs=200, **params)
    y_pred = nn_predict(model, Xv, np_, device="cpu")

    mae = val_mae(yv, y_pred)
    print(f"Done: val_mae={mae}", flush=True)

    nn_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


nn_trials_df = pd.DataFrame(nn_trials).sort_values("val_mae").reset_index(drop=True)
best_nn = {k: v for k, v in nn_trials_df.iloc[0].items() if k != "val_mae"}
print("\nNN trials:")
print(nn_trials_df)
print("Best NN:", best_nn)

Training NN: lr=0.001, loss_alpha=0.2


In [ ]:
xgb_grid = list(itertools.product([5, 7], [0.03, 0.1], [500, 1000]))
xgb_trials = []
for max_depth, lr, n_estimators in xgb_grid:
    params = {"max_depth": max_depth, "learning_rate": lr, "n_estimators": n_estimators}
    model, np_ = xgb_train(Xt, yt, Xv, yv, **params)
    y_pred = xgb_predict(model, Xv, np_)
    xgb_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


xgb_trials_df = pd.DataFrame(xgb_trials).sort_values("val_mae").reset_index(drop=True)
best_xgb = {k: v for k, v in xgb_trials_df.iloc[0].items() if k != "val_mae"}
best_xgb["max_depth"] = int(best_xgb["max_depth"])
best_xgb["n_estimators"] = int(best_xgb["n_estimators"])
print("XGBoost trials:")
print(xgb_trials_df)
print("Best XGBoost:", best_xgb)

Training NN: lr=0.001, loss_alpha=0.2


## Training and Eval

We train both models on the ranges mentioned earlier. Subsequently, we compare the model's prediction metrics across test data seeded from the same ranges as the training data:
- MAE
- RMSE
- Mean and median relative error
- Maximum error

Both models utilize similar feature engineering, opting for log-moneyness instead of standard $S$ and $K$ values. This is because option price $C$ increases roughly monotonically with $\log \left ( \frac{S}{K} \right ).$ For true predictions, we can mathematically compute log moneyness manually.
Furthermore, the NN uses a hybridized loss function, linearly interpolating standard MSE with a $y-$weighted MSE. This is done to minimize both relative error at OTM, and absolute error at ITM.

In [3]:
import pandas as pd
from src.training.train_nn import train, set_seed, get_device, predict
from src.eval.validate import preprocess_test_data, compute_metrics

set_seed(42) 
device = get_device(use_cpu=True)

results = []

for region in bundles:
    # Train only on calls, for now
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call
    
    nn_model, norm_params = train(
        X_train, y_train, X_val, y_val,
        device=device,
        patience=25,
        epochs=400,
        # **best_nn,
    )

    X_test, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test)
    y_pred = predict(nn_model, X_test, norm_params, device="cpu")

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    results.append({
        "model" : "NN",
        "regime" : region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

Epoch 0, Training Loss: 0.63869721, Eval Loss: 0.57041103
Epoch 10, Training Loss: 0.33823305, Eval Loss: 0.31740654
Epoch 20, Training Loss: 0.22515064, Eval Loss: 0.20871353
Epoch 30, Training Loss: 0.14410846, Eval Loss: 0.13683720
Epoch 40, Training Loss: 0.10014478, Eval Loss: 0.09586947
Epoch 50, Training Loss: 0.07340041, Eval Loss: 0.07066958
Epoch 60, Training Loss: 0.05586142, Eval Loss: 0.05364238
Epoch 70, Training Loss: 0.04274707, Eval Loss: 0.04090744
Epoch 80, Training Loss: 0.03339897, Eval Loss: 0.03180962
Epoch 90, Training Loss: 0.02647769, Eval Loss: 0.02520001
Epoch 100, Training Loss: 0.02141784, Eval Loss: 0.02048655
Epoch 110, Training Loss: 0.01779062, Eval Loss: 0.01709592
Epoch 120, Training Loss: 0.01517344, Eval Loss: 0.01462054
Epoch 130, Training Loss: 0.01324027, Eval Loss: 0.01275710
Epoch 140, Training Loss: 0.01174178, Eval Loss: 0.01132989
Epoch 150, Training Loss: 0.01056844, Eval Loss: 0.01019497
Epoch 160, Training Loss: 0.00961822, Eval Loss: 0.

Now to train the XGBoost model:

In [5]:
import pandas as pd
from src.training.train_xgboost import train as xgb_train, predict as xgb_predict
from src.eval.validate import preprocess_test_data, compute_metrics

xgb_results = []

for region in bundles:
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call

    xgb_model, norm_params = xgb_train(
        X_train, y_train, 
        X_val, y_val, 
        # **best_xgb
    )

    X_test_raw, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test_raw)
    y_pred = xgb_predict(xgb_model, X_test, norm_params)

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    xgb_results.append({
        "model": "XGBoost",
        "regime": region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

[0]	validation_0-rmse:0.94617
[50]	validation_0-rmse:0.20641
[100]	validation_0-rmse:0.09645
[150]	validation_0-rmse:0.06988
[200]	validation_0-rmse:0.05681
[250]	validation_0-rmse:0.05208
[300]	validation_0-rmse:0.04835
[350]	validation_0-rmse:0.04608
[400]	validation_0-rmse:0.04319
[450]	validation_0-rmse:0.04076
[499]	validation_0-rmse:0.03877
[0]	validation_0-rmse:0.94624
[50]	validation_0-rmse:0.12308
[100]	validation_0-rmse:0.02019
[150]	validation_0-rmse:0.00713
[200]	validation_0-rmse:0.00581
[250]	validation_0-rmse:0.00556
[300]	validation_0-rmse:0.00542
[350]	validation_0-rmse:0.00532
[400]	validation_0-rmse:0.00524
[450]	validation_0-rmse:0.00517
[499]	validation_0-rmse:0.00511
[0]	validation_0-rmse:0.96735
[50]	validation_0-rmse:0.23273
[100]	validation_0-rmse:0.13418
[150]	validation_0-rmse:0.10942
[200]	validation_0-rmse:0.09622
[250]	validation_0-rmse:0.09032
[300]	validation_0-rmse:0.08391
[350]	validation_0-rmse:0.08041
[400]	validation_0-rmse:0.07562
[450]	validation_

Finally, let's compare both!

In [6]:
results_df = pd.DataFrame(results)
xgb_results_df = pd.DataFrame(xgb_results)

print(pd.concat([results_df, xgb_results_df], ignore_index=True))

     model regime       mae      rmse  mean_relative_error  \
0       NN    ATM  0.003517  0.005818             0.032743   
1       NN    ITM  0.026863  0.038044             0.011113   
2       NN    OTM  0.000175  0.001170             0.099367   
3  XGBoost    ATM  0.003987  0.007476             0.032359   
4  XGBoost    ITM  0.006621  0.008725             0.002523   
5  XGBoost    OTM  0.000199  0.001355             0.119982   

   median_relative_error  max_error  
0               0.022057   0.049363  
1               0.007847   0.195734  
2               0.095866   0.039521  
3               0.023009   0.122990  
4               0.002212   0.062354  
5               0.099672   0.037328  


We observe that the NN and XGBoost predictions are not meaningfully different across ATM and OTM. **However, at ITM there exists around a 4.3x difference in MAE and RMSE**. This discrepancy most likely arises due to the inherent structure of the models rather than feature engineering. 

At ITM (or $S >> K$), both $D_1, D_2,$ grow very large. Thus, we approximate: $N(D_1), N(D_2) \approx 1,$ which means that
$$C = S - K e ^ {-rT}.$$ 
In our feature space $\log{\frac{C}{K}},$ this becomes low-curvature.

And although each model is trained independently within each regime, which eliminates cross-regime generalization effects, the aforementioned error persists, suggesting a structural bias within each model. Although GBTs also have this issue, the piecewise nature of its implementation means that each leaf within the model can be arbitrarily accurate, thus allowing them to reduce the structure by region and capture localized structure, in contrast to the NN, which is forced to minimize a global error. Furthermore, the exponential denormalization from feature-space $\log{\frac{C}{K}},$ to evaluation space $\frac{C}{K}$ punishes ITM prices disproportionately, thus amplifying the issue.

This is most likely why traditional implementations of Black-Scholes pricing often resort to GBTs over NNs for approximation.